# Script 09: GraphSAGE Baseline for Drug Repurposing

**Project 53 — Calibrated Uncertainty for Drug Repurposing via Conformal Prediction**

Builds a heterogeneous graph from Open Targets raw data and trains a GraphSAGE link-prediction model to predict drug-disease indications. Compares with the XGBoost baseline (test AUROC=0.906, AUPRC=0.625).

**Graph schema:**
- `drug --[mechanism]--> target` (drug_mechanism_of_action, ~14.5K edges)
- `target --[interacts]--> target` (STRING PPI, score > 0.7, human only)
- `target --[associated]--> disease` (association_by_datasource_direct, score > 0)
- `disease --[parent_of]--> disease` (disease ontology parent-child)

**Before running:** Upload `P53_data.zip` to your Google Drive root. Create it locally with:
```bash
cd '53 Conformal Drug Repurposing'
zip -r P53_data.zip \
  data/raw/drug_mechanism_of_action \
  data/raw/interaction \
  data/raw/association_by_datasource_direct \
  data/raw/disease \
  data/processed/drug_disease_labelled.parquet \
  outputs/xgboost_metrics.json
```

**Runtime:** Set to **GPU (T4)** via Runtime > Change runtime type.

## 1. Install Dependencies

In [ ]:
%%time
import subprocess, sys

def _install(pkg, pip_name=None):
    try:
        __import__(pkg)
        print(f"  {pkg}: already installed")
    except ImportError:
        name = pip_name or pkg
        print(f"  Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
        print(f"  {name}: installed")

print("Checking dependencies...")
_install("torch")
_install("torch_geometric", "torch_geometric")
_install("pyarrow")
_install("sklearn", "scikit-learn")

import torch
print("\nAll dependencies ready.")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"Memory: {mem / 1e9:.1f} GB")

## 2. Mount Google Drive & Set Up Data

In [ ]:
%%time
import os, time, zipfile
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Project root on Colab
PROJECT_ROOT = Path("/content/p53")
PROJECT_ROOT.mkdir(exist_ok=True)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check if data already extracted
required = [
    RAW_DIR / "drug_mechanism_of_action",
    RAW_DIR / "interaction",
    RAW_DIR / "association_by_datasource_direct",
    RAW_DIR / "disease",
    PROCESSED_DIR / "drug_disease_labelled.parquet",
]

if all(p.exists() for p in required):
    print("Data already present — skipping extraction.")
else:
    # Search for zip on Drive
    search_paths = [
        Path("/content/drive/MyDrive/P53_data.zip"),
        Path("/content/drive/MyDrive/Research/P53_data.zip"),
        Path("/content/drive/MyDrive/Colab/P53_data.zip"),
    ]
    zip_path = None
    for p in search_paths:
        if p.exists():
            zip_path = p
            break

    if zip_path is None:
        # Try upload
        print("P53_data.zip not found on Drive. Please upload it:")
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            zip_path = Path(list(uploaded.keys())[0])
        else:
            raise FileNotFoundError(
                "No data zip provided. Upload P53_data.zip to Drive root or use the upload widget."
            )

    print(f"Extracting {zip_path.name} ({zip_path.stat().st_size / 1e9:.2f} GB)...")
    t = time.time()
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.namelist()
        total = len(members)
        for i, member in enumerate(members):
            zf.extract(member, PROJECT_ROOT)
            if (i + 1) % 100 == 0 or i + 1 == total:
                pct = (i + 1) / total * 100
                print(f"\r  Extracting: {pct:.0f}% ({i+1:,}/{total:,})", end="", flush=True)
    elapsed = time.time() - t
    print(f"\n  Done — {total:,} files in {elapsed:.0f}s")

# Verify
print("\nData verification:")
for p in required:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p.relative_to(PROJECT_ROOT)}")

## 3. Imports & Utilities

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.transforms import ToUndirected
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

GRAPH_CACHE = PROCESSED_DIR / "hetero_graph.pt"
RANDOM_STATE = 42
STRING_SCORE_THRESHOLD = 0.7

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def fmt_elapsed(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h: return f"{h}h {m}m {s}s"
    return f"{m}m {s}s" if m else f"{s}s"


def progress_bar(current, total, width=40, prefix=""):
    frac = current / max(total, 1)
    filled = int(width * frac)
    bar = "\u2588" * filled + "\u2591" * (width - filled)
    print(f"\r  {prefix}|{bar}| {current:,}/{total:,} ({frac*100:.0f}%)", end="", flush=True)

## 4. Build Heterogeneous Graph

Constructs a graph with 3 node types (drug, target, disease) and 4 edge types from Open Targets raw data. Cached to disk after first build.

In [ ]:
%%time

def build_drug_target_edges():
    print("  [1/4] Drug-Target edges (mechanism of action)...")
    t = time.time()
    moa = pd.read_parquet(RAW_DIR / "drug_mechanism_of_action", columns=["chemblIds", "targets"])
    moa = moa.explode("chemblIds").explode("targets").dropna(subset=["chemblIds", "targets"])
    moa = moa.rename(columns={"chemblIds": "drug_id", "targets": "target_id"})
    moa = moa[["drug_id", "target_id"]].drop_duplicates()
    print(f"    {len(moa):,} edges in {fmt_elapsed(time.time()-t)}")
    return moa


def build_target_target_edges():
    print(f"  [2/4] Target-Target edges (STRING PPI, score > {STRING_SCORE_THRESHOLD})...")
    t = time.time()
    print("    Loading interaction parquet (13M+ rows, may take 1-2 min)...")
    t_load = time.time()
    ppi = pd.read_parquet(
        RAW_DIR / "interaction",
        columns=["sourceDatabase", "targetA", "targetB", "scoring", "speciesA", "speciesB"],
        filters=[("sourceDatabase", "==", "string")],
    )
    print(f"    Loaded {len(ppi):,} STRING rows in {fmt_elapsed(time.time() - t_load)}")

    print("    Filtering to human interactions (taxonId=9606)...")
    ppi["taxonA"] = ppi["speciesA"].apply(lambda x: x.get("taxonId") if isinstance(x, dict) else None)
    ppi["taxonB"] = ppi["speciesB"].apply(lambda x: x.get("taxonId") if isinstance(x, dict) else None)
    n_before = len(ppi)
    ppi = ppi[(ppi["taxonA"] == 9606) & (ppi["taxonB"] == 9606)]
    print(f"    {len(ppi):,} human edges (filtered {n_before - len(ppi):,} non-human)")

    n_before = len(ppi)
    ppi = ppi[ppi["scoring"] >= STRING_SCORE_THRESHOLD]
    print(f"    {len(ppi):,} high-confidence edges (filtered {n_before - len(ppi):,} below threshold)")

    ppi = ppi[["targetA", "targetB"]].drop_duplicates()
    ppi = ppi.rename(columns={"targetA": "src", "targetB": "dst"})
    print(f"    {len(ppi):,} unique edges in {fmt_elapsed(time.time()-t)}")
    return ppi


def build_target_disease_edges():
    print("  [3/4] Target-Disease edges (association scores)...")
    t = time.time()
    print("    Loading association parquet...")
    t_load = time.time()
    assoc = pd.read_parquet(
        RAW_DIR / "association_by_datasource_direct",
        columns=["targetId", "diseaseId", "associationScore"],
    )
    print(f"    Loaded {len(assoc):,} rows in {fmt_elapsed(time.time() - t_load)}")
    n_before = len(assoc)
    assoc = assoc[assoc["associationScore"] > 0]
    print(f"    {len(assoc):,} with score > 0 (filtered {n_before - len(assoc):,} zeros)")
    assoc = assoc[["targetId", "diseaseId"]].drop_duplicates()
    assoc = assoc.rename(columns={"targetId": "target_id", "diseaseId": "disease_id"})
    print(f"    {len(assoc):,} unique edges in {fmt_elapsed(time.time()-t)}")
    return assoc


def build_disease_disease_edges():
    print("  [4/4] Disease-Disease edges (ontology parent-child)...")
    t = time.time()
    disease_df = pd.read_parquet(RAW_DIR / "disease", columns=["id", "parents"])
    total = len(disease_df)
    print(f"    Processing {total:,} disease nodes...")
    rows = []
    for i, (_, row) in enumerate(disease_df.iterrows()):
        if row["parents"] is not None:
            for parent_id in row["parents"]:
                rows.append({"child": row["id"], "parent": parent_id})
        if (i + 1) % 10000 == 0 or i + 1 == total:
            progress_bar(i + 1, total, prefix="Diseases ")
    print()
    dd = pd.DataFrame(rows).drop_duplicates()
    print(f"    {len(dd):,} edges in {fmt_elapsed(time.time()-t)}")
    return dd


# Load labelled data
print("Loading labelled data...")
labels_df = pd.read_parquet(
    PROCESSED_DIR / "drug_disease_labelled.parquet",
    columns=["drug_id", "disease_id", "label", "split"],
)
print(f"  {len(labels_df):,} drug-disease pairs")
print(f"  Splits: {labels_df['split'].value_counts().to_dict()}")

# Build or load cached graph
if GRAPH_CACHE.exists():
    print(f"\nLoading cached graph from {GRAPH_CACHE}...")
    t = time.time()
    cached = torch.load(GRAPH_CACHE, weights_only=False)
    data = cached["graph"]
    drug_map = cached["drug_map"]
    target_map = cached["target_map"]
    disease_map = cached["disease_map"]
    print(f"  Loaded in {fmt_elapsed(time.time()-t)}")
else:
    print("\nBuilding heterogeneous graph from raw data...")
    t_total = time.time()

    drug_target = build_drug_target_edges()
    target_target = build_target_target_edges()
    target_disease = build_target_disease_edges()
    disease_disease = build_disease_disease_edges()

    # Node mappings — filter out None values before sorting
    print("\n  Creating node mappings...")
    drug_ids = sorted(
        (set(drug_target["drug_id"]) | set(labels_df["drug_id"])) - {None}
    )
    target_ids = sorted(
        (set(drug_target["target_id"]) | set(target_target["src"])
        | set(target_target["dst"]) | set(target_disease["target_id"])) - {None}
    )
    disease_ids = sorted(
        (set(target_disease["disease_id"]) | set(disease_disease["child"])
        | set(disease_disease["parent"]) | set(labels_df["disease_id"])) - {None}
    )
    drug_map = {gid: i for i, gid in enumerate(drug_ids)}
    target_map = {gid: i for i, gid in enumerate(target_ids)}
    disease_map = {gid: i for i, gid in enumerate(disease_ids)}
    print(f"  Nodes: {len(drug_map):,} drugs, {len(target_map):,} targets, {len(disease_map):,} diseases")

    # Build HeteroData
    print("\n  Constructing HeteroData...")
    data = HeteroData()
    data["drug"].num_nodes = len(drug_map)
    data["target"].num_nodes = len(target_map)
    data["disease"].num_nodes = len(disease_map)
    data["drug"].x = torch.zeros(len(drug_map), 1)
    data["target"].x = torch.zeros(len(target_map), 1)
    data["disease"].x = torch.zeros(len(disease_map), 1)

    # Drug -> Target
    print("    Drug->Target...", end=" ", flush=True)
    dt = drug_target[drug_target["drug_id"].isin(drug_map) & drug_target["target_id"].isin(target_map)]
    src = torch.tensor(dt["drug_id"].map(drug_map).values, dtype=torch.long)
    dst = torch.tensor(dt["target_id"].map(target_map).values, dtype=torch.long)
    data["drug", "mechanism", "target"].edge_index = torch.stack([src, dst])
    print(f"{len(dt):,} edges")

    # Target -> Target
    print("    Target->Target...", end=" ", flush=True)
    tt = target_target[target_target["src"].isin(target_map) & target_target["dst"].isin(target_map)]
    src = torch.tensor(tt["src"].map(target_map).values, dtype=torch.long)
    dst = torch.tensor(tt["dst"].map(target_map).values, dtype=torch.long)
    data["target", "interacts", "target"].edge_index = torch.stack([src, dst])
    print(f"{len(tt):,} edges")

    # Target -> Disease
    print("    Target->Disease...", end=" ", flush=True)
    td = target_disease[target_disease["target_id"].isin(target_map) & target_disease["disease_id"].isin(disease_map)]
    src = torch.tensor(td["target_id"].map(target_map).values, dtype=torch.long)
    dst = torch.tensor(td["disease_id"].map(disease_map).values, dtype=torch.long)
    data["target", "associated", "disease"].edge_index = torch.stack([src, dst])
    print(f"{len(td):,} edges")

    # Disease -> Disease
    print("    Disease->Disease...", end=" ", flush=True)
    dd = disease_disease[disease_disease["child"].isin(disease_map) & disease_disease["parent"].isin(disease_map)]
    src = torch.tensor(dd["child"].map(disease_map).values, dtype=torch.long)
    dst = torch.tensor(dd["parent"].map(disease_map).values, dtype=torch.long)
    data["disease", "parent_of", "disease"].edge_index = torch.stack([src, dst])
    print(f"{len(dd):,} edges")

    # Make undirected
    data = ToUndirected()(data)

    # Save cache
    torch.save({"graph": data, "drug_map": drug_map, "target_map": target_map, "disease_map": disease_map}, GRAPH_CACHE)
    print(f"\n  Graph built and cached in {fmt_elapsed(time.time()-t_total)}")

    # Clean up DataFrames to free memory
    del drug_target, target_target, target_disease, disease_disease

# Print summary
print("\nGraph summary:")
for et in data.edge_types:
    print(f"  {et}: {data[et].edge_index.shape[1]:,} edges")
print(f"  Total nodes: {data['drug'].num_nodes + data['target'].num_nodes + data['disease'].num_nodes:,}")

## 5. Model Definition

In [ ]:
class HomoGNNEncoder(nn.Module):
    """Homogeneous GNN encoder (converted to heterogeneous via to_hetero)."""
    def __init__(self, hidden_channels, out_channels, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for i in range(num_layers):
            in_ch = (-1) if i == 0 else hidden_channels
            out_ch = out_channels if i == num_layers - 1 else hidden_channels
            self.convs.append(SAGEConv(in_ch, out_ch))
            if i < num_layers - 1:
                self.norms.append(nn.LayerNorm(out_ch))

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.norms):
                x = self.norms[i](x)
                x = F.relu(x)
                x = F.dropout(x, p=0.3, training=self.training)
        return x


class LinkPredictor(nn.Module):
    """MLP decoder for drug-disease link prediction."""
    def __init__(self, in_channels):
        super().__init__()
        self.lin1 = nn.Linear(2 * in_channels, in_channels)
        self.lin2 = nn.Linear(in_channels, 1)

    def forward(self, z_drug, z_disease):
        z = torch.cat([z_drug, z_disease], dim=-1)
        z = F.relu(self.lin1(z))
        z = F.dropout(z, p=0.3, training=self.training)
        return self.lin2(z).squeeze(-1)


class DrugRepurposingGNN(nn.Module):
    """Full model: learnable embeddings -> GraphSAGE encoder -> link predictor."""
    def __init__(self, metadata, num_drugs, num_targets, num_diseases,
                 embed_dim=128, hidden_channels=128, out_channels=128, num_layers=3):
        super().__init__()
        self.drug_emb = nn.Embedding(num_drugs, embed_dim)
        self.target_emb = nn.Embedding(num_targets, embed_dim)
        self.disease_emb = nn.Embedding(num_diseases, embed_dim)
        encoder = HomoGNNEncoder(hidden_channels, out_channels, num_layers)
        self.encoder = to_hetero(encoder, metadata, aggr="mean")
        self.predictor = LinkPredictor(out_channels)

    def encode(self, data):
        x_dict = {
            "drug": self.drug_emb.weight,
            "target": self.target_emb.weight,
            "disease": self.disease_emb.weight,
        }
        return self.encoder(x_dict, data.edge_index_dict)

    def decode(self, z_dict, drug_idx, disease_idx):
        return self.predictor(z_dict["drug"][drug_idx], z_dict["disease"][disease_idx])

    def forward(self, data, drug_idx, disease_idx):
        z_dict = self.encode(data)
        return self.decode(z_dict, drug_idx, disease_idx)


print("Model classes defined.")

## 6. Prepare Training Data

In [ ]:
%%time

print("Converting labelled pairs to tensors...")
split_data = {}
for split_name in ["train", "calibration", "test"]:
    t = time.time()
    sub = labels_df[labels_df["split"] == split_name].copy()
    n_raw = len(sub)
    mask = sub["drug_id"].isin(drug_map) & sub["disease_id"].isin(disease_map)
    sub = sub[mask]
    n_dropped = n_raw - len(sub)
    print(f"  {split_name}: {len(sub):,} pairs (dropped {n_dropped:,} not in graph)...", end=" ", flush=True)
    drug_idx = torch.tensor(sub["drug_id"].map(drug_map).values, dtype=torch.long)
    disease_idx = torch.tensor(sub["disease_id"].map(disease_map).values, dtype=torch.long)
    labels_t = torch.tensor(sub["label"].values, dtype=torch.float32)
    split_data[split_name] = (drug_idx, disease_idx, labels_t)
    n_pos = int(labels_t.sum().item())
    print(f"{n_pos:,} pos / {len(labels_t)-n_pos:,} neg — {fmt_elapsed(time.time()-t)}")

# Free labels DataFrame
del labels_df
print("\nReady for training.")

## 7. Configuration

In [ ]:
# ── Hyperparameters (adjust as needed) ──
EPOCHS = 150
HIDDEN_DIM = 128
NUM_LAYERS = 3
LEARNING_RATE = 0.001
NEG_RATIO = 5        # negative samples per positive during training
PATIENCE = 20         # early stopping patience (in epochs)

print(f"Config: {EPOCHS} epochs, dim={HIDDEN_DIM}, layers={NUM_LAYERS}, "
      f"lr={LEARNING_RATE}, neg_ratio={NEG_RATIO}, patience={PATIENCE}")

## 8. Train GraphSAGE

In [ ]:
%%time

# ── Helper functions ──

def train_epoch(model, graph, optimizer, split, dev, neg_ratio=5):
    model.train()
    drug_idx, disease_idx, labels = split
    pos_mask = labels == 1
    neg_mask = labels == 0

    pos_drug = drug_idx[pos_mask].to(dev)
    pos_disease = disease_idx[pos_mask].to(dev)
    neg_drug_lab = drug_idx[neg_mask].to(dev)
    neg_disease_lab = disease_idx[neg_mask].to(dev)

    n_extra = max(0, len(pos_drug) * neg_ratio - len(neg_drug_lab))
    if n_extra > 0:
        extra_drug = torch.randint(0, graph["drug"].num_nodes, (n_extra,), device=dev)
        extra_disease = torch.randint(0, graph["disease"].num_nodes, (n_extra,), device=dev)
        neg_drug = torch.cat([neg_drug_lab, extra_drug])
        neg_disease = torch.cat([neg_disease_lab, extra_disease])
    else:
        n_use = len(pos_drug) * neg_ratio
        perm = torch.randperm(len(neg_drug_lab))[:n_use]
        neg_drug = neg_drug_lab[perm]
        neg_disease = neg_disease_lab[perm]

    all_drug = torch.cat([pos_drug, neg_drug])
    all_disease = torch.cat([pos_disease, neg_disease])
    all_labels = torch.cat([torch.ones(len(pos_drug), device=dev),
                            torch.zeros(len(neg_drug), device=dev)])
    perm = torch.randperm(len(all_labels))
    all_drug, all_disease, all_labels = all_drug[perm], all_disease[perm], all_labels[perm]

    optimizer.zero_grad()
    n_p = all_labels.sum().item()
    pw = torch.tensor([(len(all_labels) - n_p) / max(n_p, 1)], device=dev)
    logits = model(graph.to(dev), all_drug, all_disease)
    loss = F.binary_cross_entropy_with_logits(logits, all_labels, pos_weight=pw)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, graph, split, dev):
    model.eval()
    drug_idx, disease_idx, labels = split
    drug_idx = drug_idx.to(dev)
    disease_idx = disease_idx.to(dev)
    batch_size = 100_000
    n_batches = (len(drug_idx) + batch_size - 1) // batch_size
    all_probs = []
    for i, start in enumerate(range(0, len(drug_idx), batch_size)):
        end = min(start + batch_size, len(drug_idx))
        logits = model(graph.to(dev), drug_idx[start:end], disease_idx[start:end])
        all_probs.append(torch.sigmoid(logits).cpu())
        if n_batches > 5:
            progress_bar(i + 1, n_batches, prefix="Eval ")
    if n_batches > 5:
        print()
    y_prob = torch.cat(all_probs).numpy()
    y_true = labels.numpy()
    metrics = {}
    if len(np.unique(y_true)) > 1:
        metrics["auroc"] = float(roc_auc_score(y_true, y_prob))
        metrics["auprc"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["auroc"] = float("nan")
        metrics["auprc"] = float("nan")
    metrics["brier"] = float(brier_score_loss(y_true, y_prob))
    metrics["n_samples"] = len(y_true)
    metrics["n_positive"] = int(y_true.sum())
    return metrics


# ── Build model ──

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

model = DrugRepurposingGNN(
    metadata=data.metadata(),
    num_drugs=data["drug"].num_nodes,
    num_targets=data["target"].num_nodes,
    num_diseases=data["disease"].num_nodes,
    embed_dim=HIDDEN_DIM, hidden_channels=HIDDEN_DIM,
    out_channels=HIDDEN_DIM, num_layers=NUM_LAYERS,
).to(device)

# Dummy forward pass to initialize lazy layers before counting parameters
with torch.no_grad():
    dummy_drug = torch.zeros(1, dtype=torch.long, device=device)
    dummy_disease = torch.zeros(1, dtype=torch.long, device=device)
    _ = model(data.to(device), dummy_drug, dummy_disease)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=10, min_lr=1e-6
)


# ── Training loop ──

best_cal_auroc = -1
best_epoch = -1
best_state = None
no_improve = 0
t_start = time.time()

print(f"\nTraining for up to {EPOCHS} epochs (early stopping patience={PATIENCE})...\n")

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    loss = train_epoch(model, data, optimizer, split_data["train"], device, neg_ratio=NEG_RATIO)
    ep_sec = time.time() - t_ep

    if epoch % 5 == 0 or epoch == 1:
        cal_m = evaluate(model, data, split_data["calibration"], device)
        cal_auroc = cal_m["auroc"]
        scheduler.step(cal_auroc)
        lr = optimizer.param_groups[0]["lr"]

        star = ""
        if cal_auroc > best_cal_auroc:
            best_cal_auroc = cal_auroc
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            star = " \u2605"
        else:
            no_improve += 5

        elapsed = time.time() - t_start
        eta = (elapsed / epoch) * (EPOCHS - epoch)
        pct = epoch / EPOCHS * 100
        print(
            f"  Epoch {epoch:3d}/{EPOCHS} ({pct:.0f}%) | "
            f"loss={loss:.4f} | "
            f"cal AUROC={cal_auroc:.4f} | "
            f"cal AUPRC={cal_m['auprc']:.4f} | "
            f"lr={lr:.1e} | "
            f"{ep_sec:.1f}s/ep (ETA {fmt_elapsed(eta)}){star}",
            flush=True,
        )

        if no_improve >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break
    elif epoch % 10 == 0:
        pct = epoch / EPOCHS * 100
        print(f"  Epoch {epoch:3d}/{EPOCHS} ({pct:.0f}%) | loss={loss:.4f} | {ep_sec:.1f}s/ep", flush=True)

train_time = time.time() - t_start
print(f"\nTraining complete in {fmt_elapsed(train_time)}")
print(f"Best calibration AUROC: {best_cal_auroc:.4f} at epoch {best_epoch}")

## 9. Final Evaluation & Comparison with XGBoost

In [ ]:
# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)
model.to(device)

print(f"Restored best model from epoch {best_epoch}\n")

results = {}
for split_name in ["train", "calibration", "test"]:
    if split_name == "train":
        drug_idx, disease_idx, labels_t = split_data["train"]
        n_eval = min(500_000, len(labels_t))
        perm = torch.randperm(len(labels_t))[:n_eval]
        eval_d = (drug_idx[perm], disease_idx[perm], labels_t[perm])
    else:
        eval_d = split_data[split_name]

    print(f"Evaluating {split_name}...")
    m = evaluate(model, data, eval_d, device)
    results[split_name] = m
    sfx = " (subsample)" if split_name == "train" else ""
    print(f"  {split_name}{sfx}: AUROC={m['auroc']:.4f}, AUPRC={m['auprc']:.4f}, "
          f"Brier={m['brier']:.4f} (n={m['n_samples']:,}, pos={m['n_positive']:,})\n")

# Compare with XGBoost
xgb_path = OUTPUT_DIR / "xgboost_metrics.json"
if xgb_path.exists():
    with open(xgb_path) as f:
        xgb_results = json.load(f)
    print("\n" + "=" * 60)
    print("Comparison with XGBoost (test set):")
    print("=" * 60)
    for metric in ["auroc", "auprc", "brier"]:
        xv = xgb_results.get("test", {}).get(metric, "N/A")
        gv = results["test"].get(metric, "N/A")
        if isinstance(xv, (int, float)) and isinstance(gv, (int, float)):
            diff = gv - xv
            d = "better" if (diff > 0 and metric != "brier") or (diff < 0 and metric == "brier") else "worse"
            print(f"  {metric.upper():6s}: XGBoost={xv:.4f}  GraphSAGE={gv:.4f}  (diff={diff:+.4f}, {d})")
    print("=" * 60)
else:
    print("XGBoost metrics not found — skipping comparison.")

## 10. Save Results & Download

## 9b. Export Raw Predictions for Conformal Prediction

Saves per-sample GNN predictions so conformal prediction can be applied locally.

In [ ]:
import shutil, zipfile

# Save metrics
results["training"] = {
    "epochs_run": best_epoch,
    "best_cal_auroc": best_cal_auroc,
    "training_time_sec": train_time,
    "model_params": n_params,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "learning_rate": LEARNING_RATE,
    "neg_ratio": NEG_RATIO,
    "device": str(device),
    "graph_nodes": {
        "drug": data["drug"].num_nodes,
        "target": data["target"].num_nodes,
        "disease": data["disease"].num_nodes,
    },
    "graph_edges": {
        str(et): data[et].edge_index.shape[1] for et in data.edge_types
    },
}

metrics_path = OUTPUT_DIR / "gnn_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"Metrics saved: {metrics_path}")

# Save model
model_path = OUTPUT_DIR / "graphsage_model.pt"
torch.save({
    "model_state_dict": model.cpu().state_dict(),
    "best_epoch": best_epoch,
    "results": results,
}, model_path)
print(f"Model saved: {model_path}")

# Package results
results_zip = OUTPUT_DIR / "gnn_results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for src, name in [(metrics_path, "gnn_metrics.json"), (model_path, "graphsage_model.pt")]:
        if src.exists():
            zf.write(src, name)
            print(f"  Packed: {name} ({src.stat().st_size / 1e6:.1f} MB)")
print(f"Results zip: {results_zip} ({results_zip.stat().st_size / 1e6:.1f} MB)")

# Copy to Google Drive
drive_out = Path("/content/drive/MyDrive/P53_results")
if drive_out.parent.exists():
    drive_out.mkdir(exist_ok=True)
    shutil.copy2(metrics_path, drive_out / "gnn_metrics.json")
    shutil.copy2(model_path, drive_out / "graphsage_model.pt")
    shutil.copy2(results_zip, drive_out / "gnn_results.zip")
    print(f"\nCopied to Google Drive: {drive_out}/")

# Trigger browser download
try:
    from google.colab import files
    print("\nTriggering download...")
    files.download(str(results_zip))
except Exception as e:
    print(f"Auto-download unavailable ({e}). Retrieve from Google Drive.")

In [ ]:
# Export raw GNN predictions for all splits (needed for conformal prediction analysis)
print("Exporting raw GNN predictions per split...")

gnn_preds_rows = []
for split_name in ["train", "calibration", "test"]:
    drug_idx, disease_idx, labels_t = split_data[split_name]

    # For train, subsample to keep file manageable
    if split_name == "train":
        n_use = min(500_000, len(labels_t))
        perm = torch.randperm(len(labels_t))[:n_use]
        d_idx, dis_idx, lab = drug_idx[perm], disease_idx[perm], labels_t[perm]
    else:
        d_idx, dis_idx, lab = drug_idx, disease_idx, labels_t

    # Get predictions in batches
    model.eval()
    d_idx_dev = d_idx.to(device)
    dis_idx_dev = dis_idx.to(device)
    batch_size = 100_000
    all_probs = []
    with torch.no_grad():
        for start in range(0, len(d_idx_dev), batch_size):
            end = min(start + batch_size, len(d_idx_dev))
            logits = model(data.to(device), d_idx_dev[start:end], dis_idx_dev[start:end])
            all_probs.append(torch.sigmoid(logits).cpu())
    y_prob = torch.cat(all_probs).numpy()

    # Build reverse maps
    rev_drug = {v: k for k, v in drug_map.items()}
    rev_disease = {v: k for k, v in disease_map.items()}

    for i in range(len(d_idx)):
        gnn_preds_rows.append({
            "drug_id": rev_drug[d_idx[i].item()],
            "disease_id": rev_disease[dis_idx[i].item()],
            "label": int(lab[i].item()),
            "split": split_name,
            "y_prob": float(y_prob[i]),
        })
    print(f"  {split_name}: {len(d_idx):,} predictions")

gnn_preds_df = pd.DataFrame(gnn_preds_rows)
gnn_preds_path = OUTPUT_DIR / "gnn_predictions.parquet"
gnn_preds_df.to_parquet(gnn_preds_path, index=False)
print(f"\nSaved {len(gnn_preds_df):,} predictions to {gnn_preds_path}")
print(f"File size: {gnn_preds_path.stat().st_size / 1e6:.1f} MB")

# Add to results zip for download
import zipfile
with zipfile.ZipFile(OUTPUT_DIR / "gnn_results.zip", "a") as zf:
    zf.write(gnn_preds_path, "gnn_predictions.parquet")
print("Added to gnn_results.zip")

# Copy to Drive
drive_out = Path("/content/drive/MyDrive/P53_results")
if drive_out.exists():
    import shutil
    shutil.copy2(gnn_preds_path, drive_out / "gnn_predictions.parquet")
    print(f"Copied to {drive_out}/")

# Trigger download
try:
    from google.colab import files
    files.download(str(gnn_preds_path))
except:
    print("Download from Drive manually.")